In [2]:
# financial_sentiment_pipeline.py

import numpy as np
import pytesseract
from PIL import Image, ImageEnhance
import requests
from datasets import load_dataset
from transformers import BertTokenizer, BertForSequenceClassification
import torch
import pandas as pd
import json
from lime.lime_text import LimeTextExplainer
from collections import Counter
import os
from dotenv import load_dotenv
from google import genai
import time
import logging
from functools import wraps
from tqdm import tqdm
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================================
# LOGGING CONFIGURATION
# ============================================================================

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('pipeline.log'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# ============================================================================
# DECORATORS
# ============================================================================

def retry_with_backoff(max_retries=3, initial_wait=2, backoff_factor=2):
    """Decorator for retry logic with exponential backoff"""
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            wait_time = initial_wait
            last_exception = None
            
            for attempt in range(max_retries):
                try:
                    logger.info(f"{func.__name__} - Attempt {attempt + 1}/{max_retries}")
                    return func(*args, **kwargs)
                except Exception as e:
                    last_exception = e
                    logger.warning(f"{func.__name__} failed: {str(e)}")
                    
                    if attempt < max_retries - 1:
                        logger.info(f"Waiting {wait_time}s before retry...")
                        time.sleep(wait_time)
                        wait_time *= backoff_factor
                    else:
                        logger.error(f"{func.__name__} failed after {max_retries} attempts")
            
            raise last_exception
        return wrapper
    return decorator


def validate_input(input_type='path'):
    """Decorator for input validation"""
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            if input_type == 'path' and len(args) > 0:
                path = args[0]
                if not path or not os.path.exists(path):
                    raise FileNotFoundError(f"File not found: {path}")
            elif input_type == 'api_token' and 'api_token' in kwargs:
                token = kwargs.get('api_token')
                if not token or len(token) < 10:
                    raise ValueError("Invalid API token provided")
            
            return func(*args, **kwargs)
        return wrapper
    return decorator


class RateLimiter:
    """Rate limiter for API calls"""
    def __init__(self, calls_per_minute=10):
        self.calls_per_minute = calls_per_minute
        self.min_interval = 60.0 / calls_per_minute
        self.last_call = 0
    
    def wait(self):
        """Wait if necessary to respect rate limit"""
        elapsed = time.time() - self.last_call
        if elapsed < self.min_interval:
            wait_time = self.min_interval - elapsed
            logger.info(f"Rate limiting: waiting {wait_time:.2f}s")
            time.sleep(wait_time)
        self.last_call = time.time()


# Initialize rate limiters
llama_rate_limiter = RateLimiter(calls_per_minute=10)
gemini_rate_limiter = RateLimiter(calls_per_minute=15)

# ============================================================================
# STEP 1: OCR + LLM PROCESSING
# ============================================================================

@validate_input(input_type='path')
def ocr_image(image_path):
    """Extract text from image with preprocessing and metadata"""
    logger.info(f"Starting OCR on: {image_path}")
    
    try:
        # Validate image format
        supported_formats = {'.png', '.jpg', '.jpeg', '.tiff', '.bmp'}
        file_ext = Path(image_path).suffix.lower()
        
        if file_ext not in supported_formats:
            raise ValueError(f"Unsupported format: {file_ext}. Supported: {supported_formats}")
        
        # Load and preprocess image
        image = Image.open(image_path)
        
        # Basic preprocessing
        if image.mode != 'RGB':
            image = image.convert('RGB')
        
        # Enhance contrast
        enhancer = ImageEnhance.Contrast(image)
        image = enhancer.enhance(1.5)
        
        # Extract text with confidence data
        ocr_data = pytesseract.image_to_data(image, output_type=pytesseract.Output.DICT)
        text = pytesseract.image_to_string(image)
        
        # Calculate average confidence
        confidences = [int(conf) for conf in ocr_data['conf'] if int(conf) > 0]
        avg_confidence = sum(confidences) / len(confidences) if confidences else 0
        
        metadata = {
            'image_path': image_path,
            'image_size': image.size,
            'format': file_ext,
            'character_count': len(text),
            'avg_confidence': avg_confidence,
            'low_confidence_words': sum(1 for c in confidences if c < 60)
        }
        
        logger.info(f"OCR complete - Confidence: {avg_confidence:.1f}%, Characters: {len(text)}")
        
        return text.strip(), metadata
        
    except FileNotFoundError:
        logger.error(f"Image file not found: {image_path}")
        raise
    except Exception as e:
        logger.error(f"OCR failed: {str(e)}")
        raise


@retry_with_backoff(max_retries=3, initial_wait=2)
@validate_input(input_type='api_token')
def llm_process(text, system_prompt, api_token, timeout=30):
    """Pass text through LLM with validation and error handling"""
    
    # Validate API token
    if not api_token or len(api_token) < 20:
        raise ValueError("Invalid or missing API token")
    
    # Apply rate limiting
    llama_rate_limiter.wait()
    
    logger.info(f"Sending request to LLM - Text length: {len(text)}")
    
    url = "https://router.huggingface.co/v1/chat/completions"
    
    payload = {
        "model": "meta-llama/Llama-3.1-8B-Instruct",
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": text}
        ],
        "max_tokens": 2000,
        "temperature": 0.1
    }
    
    headers = {
        "Authorization": f"Bearer {api_token}",
        "Content-Type": "application/json"
    }
    
    try:
        response = requests.post(url, headers=headers, json=payload, timeout=timeout)
        
        # Check status code
        if response.status_code != 200:
            logger.error(f"API returned status {response.status_code}: {response.text}")
            raise requests.HTTPError(f"API error: {response.status_code}")
        
        result = response.json()
        
        # Validate response structure
        if "choices" not in result or len(result["choices"]) == 0:
            raise ValueError("Invalid API response structure")
        
        content = result["choices"][0]["message"]["content"]
        logger.info(f"LLM response received - Length: {len(content)}")
        
        return content
        
    except requests.Timeout:
        logger.error("LLM request timed out")
        raise
    except requests.RequestException as e:
        logger.error(f"Request failed: {str(e)}")
        raise


@validate_input(input_type='path')
def process_user_image(image_path, api_token):
    """Process user uploaded image through OCR + LLM pipeline"""
    logger.info("="*60)
    logger.info("PROCESSING USER IMAGE")
    logger.info("="*60)
    
    intermediate_results = {}
    
    # Step 1: OCR
    logger.info("Step 1: OCR Extraction")
    ocr_text, ocr_metadata = ocr_image(image_path)
    
    intermediate_results['ocr_output'] = {
        'text': ocr_text,
        'metadata': ocr_metadata
    }
    
    print(f"\n[OCR OUTPUT]")
    print(f"Extracted {len(ocr_text)} characters")
    print(f"Average confidence: {ocr_metadata['avg_confidence']:.1f}%")
    print(f"Preview: {ocr_text[:200]}...")
    
    # Step 2: LLM cleaning and structuring
    logger.info("Step 2: LLM Cleaning and Structuring")
    system_prompt = """You are a text processing assistant. Your task is to:
1. Clean and correct any OCR errors in the provided text
2. Split the text into individual sentences
3. Return a JSON object with this structure: {"sentences": ["sentence1", "sentence2", ...]}

Only return valid JSON, no additional text or explanation."""
    
    llm_output = llm_process(ocr_text, system_prompt, api_token)
    
    intermediate_results['llm_output'] = {
        'raw': llm_output
    }
    
    # Validate JSON format
    try:
        sentences_json = json.loads(llm_output)
        
        if 'sentences' not in sentences_json or not isinstance(sentences_json['sentences'], list):
            raise ValueError("LLM output missing 'sentences' array")
        
        df = pd.DataFrame(sentences_json['sentences'], columns=['sentence'])
        
        intermediate_results['llm_output']['parsed'] = sentences_json
        intermediate_results['llm_output']['sentence_count'] = len(df)
        
        print(f"\n[LLM OUTPUT]")
        print(f"Extracted {len(df)} sentences")
        print(f"Sample sentences:")
        for i, sent in enumerate(df['sentence'].head(3), 1):
            print(f"  {i}. {sent[:100]}...")
        
        logger.info(f"Successfully extracted {len(df)} sentences")
        
        return df, intermediate_results
        
    except json.JSONDecodeError as e:
        logger.error(f"Failed to parse LLM JSON output: {str(e)}")
        logger.error(f"Raw output: {llm_output[:500]}")
        raise ValueError("LLM returned invalid JSON format")


def process_user_text(text_input, api_token):
    """Process user pasted text through LLM structuring"""
    logger.info("Processing user text input")
    
    intermediate_results = {}
    intermediate_results['raw_text'] = text_input
    
    system_prompt = """You are a text processing assistant. Your task is to:
1. Split the text into individual sentences
2. Return a JSON object with this structure: {"sentences": ["sentence1", "sentence2", ...]}

Only return valid JSON, no additional text or explanation."""
    
    llm_output = llm_process(text_input, system_prompt, api_token)
    
    intermediate_results['llm_output'] = {'raw': llm_output}
    
    try:
        sentences_json = json.loads(llm_output)
        
        if 'sentences' not in sentences_json or not isinstance(sentences_json['sentences'], list):
            raise ValueError("LLM output missing 'sentences' array")
        
        df = pd.DataFrame(sentences_json['sentences'], columns=['sentence'])
        intermediate_results['llm_output']['parsed'] = sentences_json
        
        logger.info(f"Extracted {len(df)} sentences from text input")
        return df, intermediate_results
        
    except json.JSONDecodeError:
        raise ValueError("LLM returned invalid JSON format")


def process_batch_images(image_paths, api_token):
    """Process multiple images in batch"""
    logger.info(f"Processing batch of {len(image_paths)} images")
    
    all_dfs = []
    all_intermediates = []
    
    for i, img_path in enumerate(image_paths, 1):
        logger.info(f"Processing image {i}/{len(image_paths)}")
        try:
            df, intermediates = process_user_image(img_path, api_token)
            all_dfs.append(df)
            all_intermediates.append(intermediates)
        except Exception as e:
            logger.error(f"Failed to process {img_path}: {str(e)}")
            continue
    
    if not all_dfs:
        raise ValueError("No images were successfully processed")
    
    combined_df = pd.concat(all_dfs, ignore_index=True)
    logger.info(f"Batch processing complete: {len(combined_df)} total sentences")
    
    return combined_df, all_intermediates


# ============================================================================
# STEP 2: LOAD DATASET
# ============================================================================

@retry_with_backoff(max_retries=3, initial_wait=2)
def load_financial_data():
    """Load Financial PhraseBank dataset with error handling"""
    logger.info("Loading Financial PhraseBank dataset...")
    
    try:
        dataset = load_dataset('FinanceInc/auditor_sentiment', split='train')
        
        # Validate dataset
        if len(dataset) == 0:
            raise ValueError("Downloaded dataset is empty")
        
        df = pd.DataFrame(dataset)
        
        # Validate required columns
        if 'sentence' not in df.columns or 'label' not in df.columns:
            raise ValueError("Dataset missing required columns")
        
        logger.info(f"Loaded {len(df)} sentences")
        logger.info(f"Columns: {df.columns.tolist()}")
        
        # Map to standard format
        if df['label'].dtype == 'object':
            df['label_name'] = df['label']
        else:
            label_map = {0: 'negative', 1: 'neutral', 2: 'positive'}
            df['label_name'] = df['label'].map(label_map)
        
        logger.info(f"Label distribution: {df['label_name'].value_counts().to_dict()}")
        
        return df
        
    except Exception as e:
        logger.error(f"Failed to load dataset: {str(e)}")
        raise


# ============================================================================
# STEP 3: LOAD MODEL AND RUN INFERENCE
# ============================================================================

@retry_with_backoff(max_retries=2, initial_wait=3)
def load_finbert_model():
    """Load FinBERT model and tokenizer with error handling"""
    logger.info("Loading FinBERT model...")
    
    try:
        model_name = "ProsusAI/finbert"
        tokenizer = BertTokenizer.from_pretrained(model_name)
        model = BertForSequenceClassification.from_pretrained(model_name)
        model.eval()
        
        logger.info("Model loaded successfully")
        return tokenizer, model
        
    except Exception as e:
        logger.error(f"Failed to load model: {str(e)}")
        raise


def predict_sentiment(texts, tokenizer, model, batch_size=16, confidence_threshold=None):
    """Run FinBERT inference on list of texts with progress tracking"""
    logger.info(f"Running inference on {len(texts)} samples (batch_size={batch_size})")
    
    results = []
    label_names = ['positive', 'negative', 'neutral']
    truncation_warnings = 0
    
    # Process with progress bar
    for i in tqdm(range(0, len(texts), batch_size), desc="Predicting sentiment"):
        batch_texts = texts[i:i+batch_size]
        
        for text in batch_texts:
            # Check if text will be truncated
            tokens = tokenizer.tokenize(text)
            if len(tokens) > 510:  # 512 - 2 for [CLS] and [SEP]
                truncation_warnings += 1
            
            inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512)
            
            with torch.no_grad():
                outputs = model(**inputs)
                probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
            
            pred_label_idx = torch.argmax(probs).item()
            pred_label = label_names[pred_label_idx]
            confidence = probs[0][pred_label_idx].item()
            
            result = {
                'predicted_label': pred_label,
                'confidence': confidence,
                'prob_positive': probs[0][0].item(),
                'prob_negative': probs[0][1].item(),
                'prob_neutral': probs[0][2].item()
            }
            
            # Apply confidence threshold if specified
            if confidence_threshold is None or confidence >= confidence_threshold:
                results.append(result)
            else:
                result['predicted_label'] = 'uncertain'
                results.append(result)
    
    if truncation_warnings > 0:
        logger.warning(f"{truncation_warnings} texts were truncated to 512 tokens")
    
    logger.info(f"Inference complete - {len(results)} predictions")
    return pd.DataFrame(results)


# ============================================================================
# STEP 4: EXPLAINABILITY WITH LIME
# ============================================================================

def setup_lime_explainer(tokenizer, model):
    """Create LIME explainer for FinBERT"""
    label_names = ['positive', 'negative', 'neutral']
    
    def predictor_fn(texts):
        """Wrapper function for LIME to call FinBERT"""
        predictions = []
        for text in texts:
            inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512)
            with torch.no_grad():
                outputs = model(**inputs)
                probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
            predictions.append(probs[0].numpy())
        return np.array(predictions)
    
    explainer = LimeTextExplainer(class_names=label_names)
    return explainer, predictor_fn


def explain_predictions(df, sentences, tokenizer, model, num_samples=5):
    """Generate LIME explanations with progress tracking"""
    logger.info(f"Generating LIME explanations for {num_samples} samples")
    
    explainer, predictor_fn = setup_lime_explainer(tokenizer, model)
    explanations = []
    
    # Get diverse samples
    sample_indices = []
    for label in df['predicted_label'].unique():
        label_indices = df[df['predicted_label'] == label].index.tolist()
        if label_indices:
            sample_indices.append(label_indices[0])
    
    remaining = num_samples - len(sample_indices)
    if remaining > 0:
        other_indices = [i for i in range(len(df)) if i not in sample_indices]
        sample_indices.extend(other_indices[:remaining])
    
    # Generate explanations with progress bar
    for idx in tqdm(sample_indices[:num_samples], desc="Generating LIME explanations"):
        try:
            text = sentences[idx]
            predicted_label = df.loc[idx, 'predicted_label']
            
            exp = explainer.explain_instance(
                text, 
                predictor_fn, 
                num_features=10,
                num_samples=500
            )
            
            word_weights = exp.as_list()
            
            explanations.append({
                'index': idx,
                'sentence': text,
                'predicted_label': predicted_label,
                'word_contributions': word_weights,
                'lime_object': exp
            })
            
        except Exception as e:
            logger.warning(f"Failed to generate explanation for index {idx}: {str(e)}")
            continue
    
    logger.info(f"Generated {len(explanations)} explanations")
    return explanations


def identify_interesting_samples(df, sentences, top_n=5):
    """Identify most interesting samples for explanation visualization"""
    logger.info("Identifying interesting samples for visualization")
    
    interesting_indices = []
    
    # 1. Most confident predictions per class
    for label in df['predicted_label'].unique():
        label_df = df[df['predicted_label'] == label]
        if not label_df.empty:
            top_idx = label_df.nlargest(1, 'confidence').index[0]
            interesting_indices.append({
                'index': int(top_idx),  # Convert to native Python int
                'reason': f'Most confident {label}',
                'confidence': float(df.loc[top_idx, 'confidence'])  # Convert to native Python float
            })
    
    # 2. Most uncertain predictions (near decision boundary)
    uncertain = df.nsmallest(2, 'confidence')
    for idx in uncertain.index:
        interesting_indices.append({
            'index': int(idx),  # Convert to native Python int
            'reason': 'Uncertain prediction',
            'confidence': float(df.loc[idx, 'confidence'])  # Convert to native Python float
        })
    
    # 3. If ground truth available, misclassifications
    if 'label_name' in df.columns:
        misclassified = df[df['predicted_label'] != df['label_name']]
        if not misclassified.empty:
            top_misclass = misclassified.head(2)
            for idx in top_misclass.index:
                interesting_indices.append({
                    'index': int(idx),  # Convert to native Python int
                    'reason': 'Misclassified',
                    'confidence': float(df.loc[idx, 'confidence'])  # Convert to native Python float
                })
    
    # Remove duplicates and limit
    unique_indices = []
    seen = set()
    for item in interesting_indices:
        if item['index'] not in seen:
            unique_indices.append(item)
            seen.add(item['index'])
    
    result = unique_indices[:top_n]
    logger.info(f"Identified {len(result)} interesting samples")
    
    return result


# ============================================================================
# STEP 5: ANALYSIS AND VISUALIZATION
# ============================================================================

def analyze_results(df, has_ground_truth=False):
    """Generate summary statistics and interesting findings"""
    logger.info("Analyzing results...")
    
    analysis = {
        'total_samples': len(df),
        'sentiment_distribution': df['predicted_label'].value_counts().to_dict(),
        'avg_confidence': df['confidence'].mean(),
        'high_confidence_samples': len(df[df['confidence'] > 0.9]),
        'low_confidence_samples': len(df[df['confidence'] < 0.6])
    }
    
    # Most confident predictions per class
    most_confident = {}
    for label in df['predicted_label'].unique():
        label_df = df[df['predicted_label'] == label]
        most_confident[label] = label_df.nlargest(3, 'confidence').index.tolist()
    
    analysis['most_confident_per_class'] = most_confident
    analysis['most_uncertain'] = df.nsmallest(5, 'confidence').index.tolist()
    
    # Calculate accuracy if ground truth available
    if has_ground_truth and 'label_name' in df.columns:
        accuracy = (df['predicted_label'] == df['label_name']).mean()
        analysis['accuracy'] = accuracy
        
        confusion_data = pd.crosstab(df['label_name'], df['predicted_label'])
        analysis['confusion_matrix'] = confusion_data
    
    logger.info("Analysis complete")
    return analysis

def create_visualizations(df, analysis, has_ground_truth=False, output_dir='.'):
    """Create plots and charts with error handling using Plotly"""
    logger.info("Creating visualizations...")
    
    # Ensure output directory exists
    os.makedirs(output_dir, exist_ok=True)
    
    try:
        # Sentiment distribution
        sentiment_counts = df['predicted_label'].value_counts()
        color_map = {'positive': '#2ecc71', 'negative': '#e74c3c', 'neutral': '#95a5a6'}
        
        fig1 = px.bar(
            x=sentiment_counts.index,
            y=sentiment_counts.values,
            labels={'x': 'Sentiment', 'y': 'Count'},
            title='Sentiment Distribution',
            color=sentiment_counts.index,
            color_discrete_map=color_map
        )
        fig1.update_layout(
            showlegend=False,
            title_font_size=16,
            title_font_family='Arial Black'
        )
        
        output_path = os.path.join(output_dir, 'sentiment_distribution.html')
        fig1.write_html(output_path)
        logger.info(f"Saved: {output_path}")
        
    except Exception as e:
        logger.error(f"Failed to create sentiment distribution plot: {str(e)}")
    
    try:
        # Confidence distribution
        fig2 = px.histogram(
            df,
            x='confidence',
            nbins=30,
            labels={'confidence': 'Confidence Score', 'count': 'Frequency'},
            title='Prediction Confidence Distribution'
        )
        fig2.update_traces(marker_color='#3498db', marker_line_color='black', marker_line_width=1)
        fig2.update_layout(
            title_font_size=16,
            title_font_family='Arial Black'
        )
        
        output_path = os.path.join(output_dir, 'confidence_distribution.html')
        fig2.write_html(output_path)
        logger.info(f"Saved: {output_path}")
        
    except Exception as e:
        logger.error(f"Failed to create confidence distribution plot: {str(e)}")
    
    try:
        # Confusion matrix if ground truth available
        if has_ground_truth and 'confusion_matrix' in analysis:
            confusion_data = analysis['confusion_matrix']
            
            fig3 = go.Figure(data=go.Heatmap(
                z=confusion_data.values,
                x=confusion_data.columns,
                y=confusion_data.index,
                colorscale='Blues',
                text=confusion_data.values,
                texttemplate='%{text}',
                textfont={"size": 14},
                hoverongaps=False
            ))
            
            fig3.update_layout(
                title='Confusion Matrix',
                title_font_size=16,
                title_font_family='Arial Black',
                xaxis_title='Predicted Label',
                yaxis_title='True Label'
            )
            
            output_path = os.path.join(output_dir, 'confusion_matrix.html')
            fig3.write_html(output_path)
            logger.info(f"Saved: {output_path}")
            
    except Exception as e:
        logger.error(f"Failed to create confusion matrix: {str(e)}")
    
    # Return figure objects for direct use in Streamlit
    return {
        'sentiment_dist': fig1 if 'fig1' in locals() else None,
        'confidence_dist': fig2 if 'fig2' in locals() else None,
        'confusion_matrix': fig3 if 'fig3' in locals() and has_ground_truth else None
    }


@retry_with_backoff(max_retries=3, initial_wait=2)
def gemini_analysis(analysis, df, api_key):
    """Optional: Use Gemini to generate insights with validation"""
    logger.info("Generating Gemini insights...")
    
    # Validate API key
    if not api_key or len(api_key) < 20:
        raise ValueError("Invalid or missing Gemini API key")
    
    # Apply rate limiting
    gemini_rate_limiter.wait()
    
    try:
        client = genai.Client(api_key=api_key)
        MODEL_TO_USE = "gemini-2.5-flash"
        
        summary_text = f"""
I have analyzed {analysis['total_samples']} financial sentences using FinBERT sentiment analysis.

Results:
- Sentiment Distribution: {analysis['sentiment_distribution']}
- Average Confidence: {analysis['avg_confidence']:.2%}
- High Confidence Predictions (>90%): {analysis['high_confidence_samples']}
- Low Confidence Predictions (<60%): {analysis['low_confidence_samples']}

Based on these aggregate statistics, provide 3-4 brief insights about the sentiment patterns in financial text. 
Focus on what the confidence levels and distribution might indicate about financial discourse.
Keep it concise and actionable.
"""
        
        response = client.models.generate_content(
            model=MODEL_TO_USE,
            contents=summary_text
        )
        
        logger.info("Gemini analysis complete")
        return response.text
        
    except Exception as e:
        logger.error(f"Gemini analysis failed: {str(e)}")
        raise


# ============================================================================
# MAIN PIPELINE
# ============================================================================

def run_pipeline(use_dataset=True, image_path=None, text_input=None, api_token=None, 
                gemini_api_key=None, batch_size=16, output_dir='.'):
    """Run complete pipeline with comprehensive error handling"""
    
    try:
        logger.info("="*60)
        logger.info("STARTING PIPELINE")
        logger.info("="*60)
        
        # Load model
        tokenizer, model = load_finbert_model()
        
        # Get data
        intermediate_results = None
        if use_dataset:
            df = load_financial_data()
            sentences = df['sentence'].tolist()
            has_ground_truth = True
        elif image_path:
            df, intermediate_results = process_user_image(image_path, api_token)
            sentences = df['sentence'].tolist()
            has_ground_truth = False
        elif text_input:
            df, intermediate_results = process_user_text(text_input, api_token)
            sentences = df['sentence'].tolist()
            has_ground_truth = False
        else:
            raise ValueError("Must provide either use_dataset=True, image_path, or text_input")
        
        # Run predictions
        predictions_df = predict_sentiment(sentences, tokenizer, model, batch_size=batch_size)
        df = pd.concat([df.reset_index(drop=True), predictions_df], axis=1)
        
        # Save predictions
        output_path = os.path.join(output_dir, 'predictions.csv')
        df.to_csv(output_path, index=False)
        logger.info(f"Saved: {output_path}")
        
        # Identify interesting samples for visualization
        interesting_samples = identify_interesting_samples(df, sentences, top_n=4)
        interesting_indices = [s['index'] for s in interesting_samples]
        
        # LIME explanations for interesting samples
        explanations = explain_predictions(df, sentences, tokenizer, model, num_samples=len(interesting_indices))
        
        # Save explanations
        exp_summary = []
        for exp in explanations:
            exp_summary.append({
                'index': exp['index'],
                'sentence': exp['sentence'],
                'predicted_label': exp['predicted_label'],
                'top_words': exp['word_contributions'][:5],
                'all_words': exp['word_contributions']
            })
        
        output_path = os.path.join(output_dir, 'explanations.json')
        with open(output_path, 'w') as f:
            json.dump(exp_summary, f, indent=2)
        logger.info(f"Saved: {output_path}")
        
        # Save interesting samples info
        output_path = os.path.join(output_dir, 'interesting_samples.json')
        with open(output_path, 'w') as f:
            json.dump(interesting_samples, f, indent=2)
        logger.info(f"Saved: {output_path}")
        
        # Analysis
        analysis = analyze_results(df, has_ground_truth)
        
        # Visualizations
        create_visualizations(df, analysis, has_ground_truth, output_dir)
        
        # Print summary
        print("\n" + "="*60)
        print("PIPELINE RESULTS SUMMARY")
        print("="*60)
        print(f"Total Samples: {analysis['total_samples']}")
        print(f"Sentiment Distribution: {analysis['sentiment_distribution']}")
        print(f"Average Confidence: {analysis['avg_confidence']:.2%}")
        
        if has_ground_truth and 'accuracy' in analysis:
            print(f"Accuracy: {analysis['accuracy']:.2%}")
        
        print(f"\nMost Uncertain Predictions (indices): {analysis['most_uncertain']}")
        
        print("\n" + "="*60)
        print("INTERESTING SAMPLES FOR VISUALIZATION")
        print("="*60)
        for sample in interesting_samples:
            print(f"\nIndex: {sample['index']}")
            print(f"Reason: {sample['reason']}")
            print(f"Confidence: {sample['confidence']:.2%}")
        
        print("\n" + "="*60)
        print("SAMPLE LIME EXPLANATIONS")
        print("="*60)
        for exp in explanations[:3]:
            print(f"\nSentence: {exp['sentence'][:100]}...")
            print(f"Predicted: {exp['predicted_label']}")
            print(f"Top Contributing Words:")
            for word, weight in exp['word_contributions'][:5]:
                print(f"  - '{word}': {weight:+.3f}")
        
        # Optional Gemini insights
        gemini_insights = None
        if gemini_api_key:
            try:
                gemini_insights = gemini_analysis(analysis, df, gemini_api_key)
                print("\n" + "="*60)
                print("GEMINI AI INSIGHTS")
                print("="*60)
                print(gemini_insights)
                
                output_path = os.path.join(output_dir, 'gemini_insights.txt')
                with open(output_path, 'w') as f:
                    f.write(gemini_insights)
                logger.info(f"Saved: {output_path}")
            except Exception as e:
                logger.warning(f"Gemini analysis skipped due to error: {str(e)}")
        
        # Save intermediate results if available (for Streamlit visualization)
        if intermediate_results:
            output_path = os.path.join(output_dir, 'intermediate_results.json')
            with open(output_path, 'w') as f:
                json.dump(intermediate_results, f, indent=2, default=str)
            logger.info(f"Saved: {output_path}")
        
        logger.info("="*60)
        logger.info("PIPELINE COMPLETE")
        logger.info("="*60)
        
        return {
            'dataframe': df,
            'explanations': explanations,
            'analysis': analysis,
            'interesting_samples': interesting_samples,
            'intermediate_results': intermediate_results,
            'gemini_insights': gemini_insights
        }
        
    except Exception as e:
        logger.error(f"Pipeline failed: {str(e)}")
        logger.error("Attempting to save partial results...")
        
        # Try to save whatever we have
        try:
            if 'df' in locals():
                df.to_csv(os.path.join(output_dir, 'partial_predictions.csv'), index=False)
                logger.info("Saved partial predictions")
        except:
            pass
        
        raise


# ============================================================================
# EXECUTION
# ============================================================================

if __name__ == "__main__":
    # Configuration
    USE_DATASET = True  # Set to False to use image upload or text input
    IMAGE_PATH = None  # Set path if using image
    TEXT_INPUT = None  # Set text if using direct text input
    BATCH_SIZE = 16  # Adjust based on available memory
    OUTPUT_DIR = './output'
    
    # Load API keys from environment
    load_dotenv()
    LLAMA_API_TOKEN = os.getenv("HUGGINGFACE_API_TOKEN")
    GEMINI_API_KEY = os.getenv("GOOGLE_API_KEY")
    
    # Create output directory
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    # Run pipeline
    results = run_pipeline(
        use_dataset=USE_DATASET,
        image_path=IMAGE_PATH,
        text_input=TEXT_INPUT,
        api_token=LLAMA_API_TOKEN,
        gemini_api_key=GEMINI_API_KEY,
        batch_size=BATCH_SIZE,
        output_dir=OUTPUT_DIR
    )
    
    print("\n" + "="*60)
    print("GENERATED FILES")
    print("="*60)
    print(f"Output directory: {OUTPUT_DIR}")
    print("- predictions.csv")
    print("- explanations.json")
    print("- interesting_samples.json")
    print("- sentiment_distribution.png")
    print("- confidence_distribution.png")
    if results['intermediate_results']:
        print("- intermediate_results.json")
    if 'label_name' in results['dataframe'].columns:
        print("- confusion_matrix.png")
    if results['gemini_insights']:
        print("- gemini_insights.txt")
    print("- pipeline.log")

2025-12-01 23:23:58,877 - __main__ - INFO - ============================================================
2025-12-01 23:23:58,879 - __main__ - INFO - STARTING PIPELINE
2025-12-01 23:23:58,882 - __main__ - INFO - ============================================================
2025-12-01 23:23:58,884 - __main__ - INFO - load_finbert_model - Attempt 1/2
2025-12-01 23:23:58,886 - __main__ - INFO - Loading FinBERT model...
2025-12-01 23:24:01,212 - __main__ - INFO - Model loaded successfully
2025-12-01 23:24:01,212 - __main__ - INFO - load_financial_data - Attempt 1/3
2025-12-01 23:24:01,220 - __main__ - INFO - Loading Financial PhraseBank dataset...
2025-12-01 23:24:02,940 - __main__ - INFO - Loaded 3877 sentences
2025-12-01 23:24:02,950 - __main__ - INFO - Columns: ['sentence', 'label']
2025-12-01 23:24:02,960 - __main__ - INFO - Label distribution: {'neutral': 2320, 'positive': 1077, 'negative': 480}
2025-12-01 23:24:02,960 - __main__ - INFO - Running inference on 3877 samples (batch_size=16


PIPELINE RESULTS SUMMARY
Total Samples: 3877
Sentiment Distribution: {'neutral': 2066, 'positive': 1229, 'negative': 582}
Average Confidence: 87.45%
Accuracy: 88.88%

Most Uncertain Predictions (indices): [3395, 2506, 2182, 313, 3823]

INTERESTING SAMPLES FOR VISUALIZATION

Index: 3741
Reason: Most confident positive
Confidence: 96.24%

Index: 2423
Reason: Most confident neutral
Confidence: 95.82%

Index: 350
Reason: Most confident negative
Confidence: 97.73%

Index: 3395
Reason: Uncertain prediction
Confidence: 38.32%

SAMPLE LIME EXPLANATIONS

Sentence: Altia 's operating profit jumped to EUR 47 million from EUR 6.6 million ....
Predicted: positive
Top Contributing Words:
  - 'jumped': -0.024
  - 'to': -0.023
  - 'from': +0.019
  - 'EUR': -0.019
  - 's': -0.017

Sentence: Vaisala , headquartered in Helsinki in Finland , develops and manufactures electronic measurement sy...
Predicted: neutral
Top Contributing Words:
  - 'develops': -0.004
  - 'manufactures': -0.002
  - 'and': +0.002